In [ ]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("stats_file.xlsx")
daily = pd.read_excel("daily_demand.xlsx")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle_Time"]) * data["Cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand_tomorrow = row["Actual_Tomorrow"]
    tentative_future = row["Tentative_Future"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    for qty in np.arange(0, 5000, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

data.to_excel("daily_plan.xlsx", index=False)

print("✅ Optimization complete")

In [ ]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

SETUP_TIME = 40  # minutes
SETUP_HOURS = SETUP_TIME / 60

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="Sheet2")
daily = pd.read_excel("C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx", sheet_name="Sheet1")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["Sum of 28-02-2026"]
    tentative_future = row["Sum of 02-03-2026"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    max_qty = row["Feb INDENT"] * 1.2

    for qty in np.arange(0, max_qty + 1, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        # Setup penalty (only if production happens)
        setup_penalty = rate * SETUP_HOURS if qty > 0 else 0

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory +
            setup_penalty
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

# ============================
# TIME REQUIRED COLUMN
# ============================

data["Time Required (hrs)"] = data["Planned_Qty"] / data["Rate"]

# ============================
# SAVE OUTPUT
# ============================

data.to_excel("daily_plan.xlsx", index=False)

print("✅ Optimization complete")

In [ ]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

SETUP_TIME = 40  # minutes
SETUP_HOURS = SETUP_TIME / 60

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="Sheet2")
daily = pd.read_excel("C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx", sheet_name="Sheet1")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["Sum of 28-02-2026"]
    tentative_future = row["Sum of 02-03-2026"]
    std = row["Std_Deviation"]
    rate = row["Rate"]
    indent = row["Feb INDENT"]

    best_qty = 0
    best_cost = np.inf

    # Avoid overproduction
    target_stock = demand_tomorrow + tentative_future + std
    max_qty = max(0, target_stock - inventory)
    max_qty = min(max_qty, indent * 1.2)

    for qty in np.arange(0, max_qty + 1, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.maximum(
            0,
            np.random.normal(tentative_future, std, SIMULATIONS)
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        setup_penalty = rate * SETUP_HOURS if qty > 0 else 0

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory +
            setup_penalty
        )

        # Protect tomorrow
        tomorrow_shortage = max(0, demand_tomorrow - (inventory + qty))
        cost += SHORTAGE_PENALTY * tomorrow_shortage * 2

        # Setup consumes time
        hours_needed = (qty / rate + SETUP_HOURS) if qty > 0 else 0

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    # ================================
    # STRATEGIC CLOSE LOGIC
    # ================================

    full_time = indent / rate + SETUP_HOURS

    if full_time <= AVAILABLE_HOURS:

        remaining = max(0, indent - (inventory + best_qty))

        if tentative_future > 0:
            future_runs = remaining / tentative_future
        else:
            future_runs = 0

        setup_saving = future_runs * rate * SETUP_HOURS

        strategic_bonus = setup_saving * 0.3

        full_cost = HOLDING_PENALTY * max(0, indent - (inventory + demand_tomorrow))

        adjusted_full_cost = full_cost - strategic_bonus

        if adjusted_full_cost < best_cost:
            best_qty = indent
            best_cost = adjusted_full_cost

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

# ============================
# TIME REQUIRED COLUMN
# ============================

data["Time Required (hrs)"] = np.where(
    data["Planned_Qty"] > 0,
    (data["Planned_Qty"] / data["Rate"]) + SETUP_HOURS,
    0
)

# ============================
# SAVE OUTPUT
# ============================

data.to_excel("daily_plan.xlsx", index=False)

print("✅ Optimization complete")

In [ ]:
import pandas as pd
import numpy as np

# ============================
# LOAD DATA
# ============================

df = pd.read_excel("production_calculated.xlsx")

df.fillna(0, inplace=True)

# ============================
# 1. PLAN vs ACTUAL
# ============================

df["Variance"] = df["Produced_Qty"] - df["Planned_Qty"]
df["%Variance"] = np.where(
    df["Planned_Qty"] == 0,
    0,
    (df["Variance"] / df["Planned_Qty"]) * 100
)

# ============================
# 2. QUALITY
# ============================

df["Good_Qty"] = df["Produced_Qty"] - df["Rejected_Qty"]
df["Yield"] = np.where(
    df["Produced_Qty"] == 0,
    0,
    df["Good_Qty"] / df["Produced_Qty"]
)

# ============================
# 3. CYCLE PERFORMANCE
# ============================

df["Cycle_Efficiency"] = np.where(
    df["Actual_Cycle_Time"] == 0,
    0,
    df["Target_Cycle_Time"] / df["Actual_Cycle_Time"]
)

# ============================
# 4. AVAILABLE TIME
# ============================

df["Available_Time"] = 24 - df["Downtime"] - df["Break_Time"]

# ============================
# 5. THROUGHPUT
# ============================

df["Actual_Output_Rate"] = np.where(
    df["Available_Time"] == 0,
    0,
    df["Produced_Qty"] / df["Available_Time"]
)

df["Ideal_Output_Rate"] = 3600 / df["Target_Cycle_Time"]

# ============================
# 6. INDENT FULFILLMENT
# ============================

indent_summary = df.groupby("Indent_ID").agg({
    "Produced_Qty": "sum",
    "Indent_Qty": "first"
}).reset_index()

indent_summary["Fulfillment_%"] = (
    indent_summary["Produced_Qty"] /
    indent_summary["Indent_Qty"]
) * 100

indent_summary["Status"] = np.where(
    indent_summary["Fulfillment_%"] >= 100,
    "Complete",
    np.where(indent_summary["Fulfillment_%"] >= 50,
             "Partial",
             "At Risk")
)

# ============================
# 7. DAYS COVERAGE
# ============================

df["Days_Coverage"] = np.where(
    df["Daily_Demand"] == 0,
    0,
    df["Good_Qty"] / df["Daily_Demand"]
)

# ============================
# 8. MACHINE SUMMARY
# ============================

machine_summary = df.groupby("Machine").agg({
    "Planned_Qty": "sum",
    "Produced_Qty": "sum",
    "Good_Qty": "sum",
    "Downtime": "sum"
}).reset_index()

machine_summary["Efficiency"] = (
    machine_summary["Produced_Qty"] /
    machine_summary["Planned_Qty"]
)

# ============================
# 9. DAILY SUMMARY
# ============================

daily_summary = df.groupby("Date").agg({
    "Planned_Qty": "sum",
    "Produced_Qty": "sum",
    "Rejected_Qty": "sum",
    "Downtime": "sum"
}).reset_index()

# ============================
# SAVE EXCEL REPORT
# ============================

with pd.ExcelWriter("Production_Analysis_Report.xlsx") as writer:
    df.to_excel(writer, sheet_name="Detailed_Data", index=False)
    machine_summary.to_excel(writer, sheet_name="Machine_Summary", index=False)
    daily_summary.to_excel(writer, sheet_name="Daily_Summary", index=False)
    indent_summary.to_excel(writer, sheet_name="Indent_Status", index=False)

print("✅ Excel Report Generated")

# ============================
# SAVE DASHBOARD DATA
# ============================

df.to_csv("Dashboard_Detailed.csv", index=False)
machine_summary.to_csv("Dashboard_Machine.csv", index=False)
daily_summary.to_csv("Dashboard_Daily.csv", index=False)
indent_summary.to_csv("Dashboard_Indent.csv", index=False)

print("✅ Dashboard Files Generated")